# Quality Check: SLEAP-DANNCE Keypoint Tracking

This notebook assesses the quality and consistency of 3D keypoint tracking from SLEAP and DANNCE.

**Analyses:**
1. Linear alignment of SLEAP and DANNCE keypoints (Procrustes, fit on 10 random frames)
2. Per-keypoint distance tracking within a session
3. Per-keypoint distance tracking across sessions (with calibration boundaries)
4. Temporal alignment quality
5. Bone length consistency (tracking stability)
6. Keypoint jitter and dropout detection
7. QC video generation (2D overlay + 3D comparison)

In [33]:
%matplotlib qt

import matplotlib.pyplot as plt

In [34]:
import sys
sys.path.insert(0, '.')

import numpy as np

from config import NODES, EDGES, CALIBRATION_DATES, DATA_ROOT, calibration_path
from data_io import (
    load_session_df, load_sleap_dannce_keys, load_aligned_data,
    load_dannce_predictions, load_sleap_keys_3d,
)
from qc_utils import (
    find_sleap_dannce_alignment, compute_per_keypoint_distances,
    summarize_keypoint_distances, compute_session_qc_summary,
    track_distances_across_sessions, compute_temporal_alignment_quality,
    compute_keypoint_jitter, detect_tracking_dropouts,
    compute_bone_length_consistency,
    plot_per_keypoint_distances, plot_distance_over_time,
    plot_multi_session_distances, plot_temporal_alignment,
    generate_qc_video,
    test_calibration_epoch_effect, compute_days_since_calibration,
    plot_calibration_epoch_effect, plot_days_since_calibration,
    plot_snout_error_spatial_density,
)

from scipy.ndimage import median_filter



## Configuration

In [44]:
# --- Choose a single session for detailed QC ---
rat = 'R1'
session = '2025_12_02_1'
calibration_dates = ['2025_07_27', '2025_11_06', '2025_11_13', '2025_12_08', '2026_02_06']

# --- Calibration dates (fill in with actual calibration folder names) ---
# These mark boundaries in the multi-session tracking plot
#calibration_dates = CALIBRATION_DATES  # or override: ['2025_07_27', '2025_09_15', ...]

## 1. Single-session: SLEAP-DANNCE alignment

In [45]:
# Load data
keys = load_sleap_dannce_keys(rat, session, fmt='mat')
aligned = load_aligned_data(rat, session, fmt='mat')

sleap_3d = median_filter(keys['sleap_keys_3D'], size=(11, 1, 1))
dannce_3d = keys['dannce_keys_3D']

# Handle DANNCE shape if needed
if dannce_3d.ndim == 4:
    dannce_3d = dannce_3d.squeeze(axis=1).transpose(0, 2, 1)
else:
    dannce_3d = dannce_3d.transpose(0, 2, 1)

dannce_3d = median_filter(dannce_3d, size=(25, 1, 1))

aligned_indices = aligned['dannce_idx_for_sleap_cams'].astype(int).ravel()

print(f'SLEAP shape: {sleap_3d.shape}')
print(f'DANNCE shape: {dannce_3d.shape}')
print(f'Aligned indices: {len(aligned_indices)}')

SLEAP shape: (36000, 23, 3)
DANNCE shape: (92500, 23, 3)
Aligned indices: 36001


In [46]:
# Find best alignment (Procrustes on 10 random frames, tries z-flip automatically)
alignment = find_sleap_dannce_alignment(
    sleap_3d, dannce_3d, aligned_indices,
    n_sample_frames=10, seed=42, try_z_flip=True
)

print(f"Alignment residual: {alignment['residual']:.2f} calibration units")
print(f"Z-flipped: {alignment['z_flipped']}")
print(f"Scale factor: {alignment['s']:.4f}")
print(f"Rotation matrix:\n{alignment['R']}")
print(f"Translation: {alignment['t']}")

Alignment residual: 11.66 calibration units
Z-flipped: True
Scale factor: 1.0133
Rotation matrix:
[[-9.99854064e-01 -4.53421938e-04 -1.70776030e-02]
 [ 5.23976497e-04 -9.99991346e-01 -4.12716094e-03]
 [-1.70755839e-02 -4.13550690e-03  9.99845649e-01]]
Translation: [-1.18892877 -4.50802533 -4.19276958]


In [47]:
session_list = []

for rat in ['R1', 'R2', 'R3']:

    df = load_session_df()
    rat_sessions = df[df['rat'] == rat]['session'].tolist()

    session_list = session_list + [(rat, s) for s in rat_sessions]

    print(f'Will process {len(session_list)} sessions for {rat}')



Will process 73 sessions for R1
Will process 148 sessions for R2
Will process 227 sessions for R3


In [39]:
i = 0
import os
for rat, session in session_list:
    if not i%10 == 0:
        i +=1
        continue
    # Load data
    keys = load_sleap_dannce_keys(rat, session, fmt='mat')
    aligned = load_aligned_data(rat, session, fmt='mat')

    sleap_3d = median_filter(keys['sleap_keys_3D'], size=(11, 1, 1))
    dannce_3d = keys['dannce_keys_3D']

    # Handle DANNCE shape if needed
    if dannce_3d.ndim == 4:
        dannce_3d = dannce_3d.squeeze(axis=1).transpose(0, 2, 1)
    else:
        dannce_3d = dannce_3d.transpose(0, 2, 1)

    dannce_3d = median_filter(dannce_3d, size=(25, 1, 1))

    aligned_indices = aligned['dannce_idx_for_sleap_cams'].astype(int).ravel()

    # Find best alignment (Procrustes on 10 random frames, tries z-flip automatically)
    alignment = find_sleap_dannce_alignment(
        sleap_3d, dannce_3d, aligned_indices,
        n_sample_frames=10, seed=42, try_z_flip=True
    )

    if alignment['residual'] > 40:
        continue

    output_video = os.path.join('/home/yutaka-sprague/CLIRB/data/sleap_dannce_keys_2026_02_18',rat, session, 'qc_video.mp4')

    #generate_qc_video(
    #    rat, session,
    #    output_path=output_video,
    #    alignment_result=alignment,
    #    n_frames=3000,
    #    start_frame=2000,
    #    fps=20,
    #    camera_idx=1,  # Camera1
    #    render_3d = True
    #)

    i +=1

    


KeyboardInterrupt: 

## 2. Per-keypoint distances (single session)

In [48]:
# Apply alignment and compute distances
sleap_aligned = alignment['apply'](sleap_3d)
distances = compute_per_keypoint_distances(sleap_aligned, dannce_3d, aligned_indices)
summary = summarize_keypoint_distances(distances)

print(f"Overall mean distance: {summary['overall_mean']:.2f} calibration units")
print(f"Overall median distance: {summary['overall_median']:.2f} calibration units")
print(f"Valid frames: {summary['n_valid_frames']}")

Overall mean distance: 10.95 calibration units
Overall median distance: 9.28 calibration units
Valid frames: 36000


In [50]:
# Per-keypoint bar chart
fig, ax = plt.subplots(figsize=(14, 5))
plot_per_keypoint_distances(summary, ax=ax, title=f'{rat}/{session} — Per-keypoint distance')
plt.tight_layout()
plt.show()

In [51]:
# Distance over time (rolling mean)
fig, ax = plt.subplots(figsize=(14, 4))
plot_distance_over_time(
    distances, 
    keypoint_indices=[0, 3, 4, 10, 14, 18, 22],  # Snout, SpineF, SpineM, HandL, HandR, FootL, FootR
    window=200, ax=ax,
    title=f'{rat}/{session} — Keypoint distance over time'
)
plt.tight_layout()
plt.show()

## 3. Temporal alignment quality

In [52]:
temporal = compute_temporal_alignment_quality(aligned)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_temporal_alignment(temporal, ax=axes[0], title=f'{rat}/{session}')
plt.tight_layout()
plt.show()

print(f"Mean time diff: {temporal['mean_time_diff_ms']:.1f} ms")
print(f"Median time diff: {temporal['median_time_diff_ms']:.1f} ms")
print(f"Max time diff: {temporal['max_time_diff_ms']:.1f} ms")
print(f"SLEAP frame interval: mean={np.mean(temporal['sleap_frame_intervals_ms']):.1f} ms, "
      f"std={np.std(temporal['sleap_frame_intervals_ms']):.1f} ms")
print(f"DANNCE frame interval: mean={np.mean(temporal['dannce_frame_intervals_ms']):.1f} ms, "
      f"std={np.std(temporal['dannce_frame_intervals_ms']):.1f} ms")

Mean time diff: 5.0 ms
Median time diff: 5.0 ms
Max time diff: 10.0 ms
SLEAP frame interval: mean=50.0 ms, std=3.8 ms
DANNCE frame interval: mean=20.0 ms, std=0.1 ms


## 4. Bone length consistency

In [53]:
# SLEAP bone lengths
sleap_bone_lengths, sleap_bone_stats = compute_bone_length_consistency(sleap_aligned)

# DANNCE bone lengths (use matched frames)
valid_idx = aligned_indices[aligned_indices < len(dannce_3d)]
dannce_bone_lengths, dannce_bone_stats = compute_bone_length_consistency(dannce_3d[valid_idx])

# Compare coefficient of variation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(len(EDGES))
labels = sleap_bone_stats['edge_labels']

axes[0].bar(x - 0.2, sleap_bone_stats['cv'], 0.4, label='SLEAP', alpha=0.7, color='cyan')
axes[0].bar(x + 0.2, dannce_bone_stats['cv'], 0.4, label='DANNCE', alpha=0.7, color='magenta')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=90, fontsize=7)
axes[0].set_ylabel('Coefficient of Variation')
axes[0].set_title('Bone length CV (lower = more consistent)')
axes[0].legend()

axes[1].bar(x - 0.2, sleap_bone_stats['mean'], 0.4, 
            yerr=sleap_bone_stats['std'], label='SLEAP', alpha=0.7, color='cyan', capsize=2)
axes[1].bar(x + 0.2, dannce_bone_stats['mean'], 0.4,
            yerr=dannce_bone_stats['std'], label='DANNCE', alpha=0.7, color='magenta', capsize=2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=90, fontsize=7)
axes[1].set_ylabel('Bone length (calibration units)')
axes[1].set_title('Mean bone lengths')
axes[1].legend()

plt.suptitle(f'{rat}/{session} — Bone length consistency')
plt.tight_layout()
plt.show()

## 5. Jitter and dropout detection

In [60]:
# SLEAP jitter
sleap_jitter = compute_keypoint_jitter(sleap_3d, window=5)
dannce_matched = dannce_3d[aligned_indices[aligned_indices < len(dannce_3d)]]
dannce_jitter = compute_keypoint_jitter(dannce_matched, window=5)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ki in [0, 3, 4, 10, 14]:  # Snout, SpineF, SpineM, HandL, HandR
    axes[0].plot(sleap_jitter[:, ki], alpha=0.5, label=NODES[ki])
axes[0].set_title(f'SLEAP keypoint jitter — {rat}/{session}')
axes[0].set_ylabel('Jitter (calibration units)')
axes[0].legend(fontsize=8)

for ki in [0, 3, 4, 10, 14]:
    axes[1].plot(dannce_jitter[:, ki], alpha=0.5, label=NODES[ki])
axes[1].set_title(f'DANNCE keypoint jitter')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Jitter (calibration units)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [55]:
# Dropout detection
sleap_dropouts = detect_tracking_dropouts(sleap_3d, max_displacement=100)
dannce_dropouts = detect_tracking_dropouts(dannce_matched, max_displacement=100)

print(f'SLEAP dropout frames: {np.sum(sleap_dropouts.any(axis=1))} / {len(sleap_dropouts)}')
print(f'DANNCE dropout frames: {np.sum(dannce_dropouts.any(axis=1))} / {len(dannce_dropouts)}')

# Which keypoints have the most dropouts?
sleap_dropout_counts = sleap_dropouts.sum(axis=0)
dannce_dropout_counts = dannce_dropouts.sum(axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(NODES))
ax.bar(x - 0.2, sleap_dropout_counts, 0.4, label='SLEAP', alpha=0.7, color='cyan')
ax.bar(x + 0.2, dannce_dropout_counts, 0.4, label='DANNCE', alpha=0.7, color='magenta')
ax.set_xticks(x)
ax.set_xticklabels(NODES, rotation=45, ha='right')
ax.set_ylabel('Dropout count')
ax.set_title(f'Tracking dropouts (>100mm jump) — {rat}/{session}')
ax.legend()
plt.tight_layout()
plt.show()

SLEAP dropout frames: 1 / 36000
DANNCE dropout frames: 1 / 36001


## 6. Multi-session distance tracking

In [27]:
rat = 'R2'

In [28]:
# Load session list and select sessions for a given rat
df = load_session_df()
rat_sessions = df[df['rat'] == rat]['session'].tolist()

session_list = [(rat, s) for s in rat_sessions]
print(f'Will process {len(session_list)} sessions for {rat}')

Will process 75 sessions for R2


In [29]:
# Track distances across all sessions (this may take a while)
qc_results = track_distances_across_sessions(session_list, n_sample_frames=10, seed=42)

Processing R2/2025_09_29_1...
Processing R2/2025_09_30_1...
Processing R2/2025_10_02_2...
Processing R2/2025_10_28_2...
Processing R2/2025_10_29_2...
Processing R2/2025_10_30_1...
Processing R2/2025_10_31_2...
Processing R2/2025_11_01_2...
Processing R2/2025_11_03_2...
Processing R2/2025_11_04_1...
Processing R2/2025_11_06_2...
Processing R2/2025_11_07_2...
Processing R2/2025_11_08_2...
Processing R2/2025_11_09_2...
Processing R2/2025_11_10_2...
Processing R2/2025_11_11_2...
Processing R2/2025_11_17_2...
Processing R2/2025_11_19_2...
Processing R2/2025_11_20_2...
Processing R2/2025_11_21_2...
Processing R2/2025_11_24_2...
Processing R2/2025_11_29_2...
Processing R2/2025_12_02_2...
Processing R2/2025_12_07_2...
Processing R2/2026_02_02_2...
Processing R2/2026_02_05_2...
Processing R2/2026_02_18_2...
Processing R2/2025_09_30_2...
Processing R2/2025_10_02_1...
Processing R2/2025_10_03_2...
Processing R2/2025_10_28_1...
Processing R2/2025_10_29_1...
Processing R2/2025_10_30_2...
Processing

In [30]:
qc_results

[{'alignment': {'R': array([[ 0.96698159,  0.18189385, -0.17849715],
          [-0.14432447,  0.96812745,  0.20469414],
          [ 0.2100406 , -0.17217396,  0.96241315]]),
   's': np.float64(0.2394870588377018),
   't': array([ 85.74505109, 133.08404131,  32.80420161]),
   'residual': np.float64(274.61358781464855),
   'z_flipped': True,
   'apply': <function qc_utils.find_sleap_dannce_alignment.<locals>.apply_transform(pts)>,
   'apply_inverse': <function qc_utils.find_sleap_dannce_alignment.<locals>.apply_inverse(pts)>},
  'distances': array([[421.93163028, 396.45484307, 374.63136476, ..., 335.3065307 ,
          331.52861628, 331.8186938 ],
         [340.53963125, 356.751489  , 334.85285457, ..., 329.36179864,
          328.14596687, 332.3342362 ],
         [320.10198912, 345.47746837, 321.20025471, ..., 327.49678149,
          328.26578669, 329.95402898],
         ...,
         [361.34639588, 383.38500775, 360.71236374, ..., 336.70627633,
          344.88209435, 335.13429723],
   

In [31]:
#%matplotlib qt
# Plot overall distance across sessions (sorted chronologically, outliers filtered)
from qc_utils import _sort_qc_results_chronologically

outlier_threshold = 50  # sessions with mean distance > this are excluded from plot

fig, ax = plt.subplots(figsize=(14, 5))
plot_multi_session_distances(
    qc_results, calibration_dates=calibration_dates, ax=ax,
    outlier_threshold=outlier_threshold,
)
plt.show()

In [32]:
# Per-keypoint distance heatmap across sessions (sorted chronologically, outliers filtered)
import matplotlib.colors as mcolors
from qc_utils import _sort_qc_results_chronologically

outlier_threshold = 50  # same threshold as the line plot above

sorted_results = _sort_qc_results_chronologically(qc_results)
valid_results = [
    r for r in sorted_results
    if 'error' not in r and r['summary']['overall_mean'] <= outlier_threshold
]
n_excluded = sum(
    1 for r in sorted_results
    if 'error' not in r and r['summary']['overall_mean'] > outlier_threshold
)

session_labels = [f"{r['session']}" for r in valid_results]
dist_matrix = np.array([r['summary']['per_keypoint_mean'] for r in valid_results])  # (n_sessions, 23)

fig, ax = plt.subplots(figsize=(14, max(4, len(valid_results) * 0.3)))
im = ax.imshow(dist_matrix, aspect='auto', cmap='hot')
ax.set_xticks(np.arange(len(NODES)))
ax.set_xticklabels(NODES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(np.arange(len(session_labels)))
ax.set_yticklabels(session_labels, fontsize=8)
title = f'{rat} — Per-keypoint mean distance across sessions'
if n_excluded > 0:
    title += f' ({n_excluded} outlier sessions excluded)'
ax.set_title(title)
plt.colorbar(im, label='Distance (calibration units)')
plt.tight_layout()
plt.show()

## 7. QC Video generation

Generates a side-by-side video:
- **Left**: SLEAP (cyan) and DANNCE (magenta) keypoints projected onto Camera1 video
- **Right**: 3D aligned keypoints overlaid with connecting lines

In [ ]:
#2.5 minutes to render 1000 frames without 3D panel
#3.5 minutes to render 1000 frames with 3D panel
rat = 'R1'
session = '2025_10_01_2'
output_video = f'qc_video_{rat}_{session}.mp4'

generate_qc_video(
    rat, session,
    output_path=output_video,
    alignment_result=alignment,
    n_frames=1000,
    start_frame=7200,
    fps=20,
    camera_idx=1,  # Camera1
    render_3d = True
)

Loading data for R1/2025_10_01_2...
Alignment residual: 18.92
Projecting SLEAP keypoints to 2D...
Transforming DANNCE to SLEAP space and projecting to 2D...
Rendering 10000 frames (with 3D panel)...
  1000/10000 frames rendered
  2000/10000 frames rendered
  3000/10000 frames rendered
  4000/10000 frames rendered
  5000/10000 frames rendered
  6000/10000 frames rendered
  7000/10000 frames rendered
  8000/10000 frames rendered
  9000/10000 frames rendered
  10000/10000 frames rendered
Saved to qc_video_R1_2025_10_01_2.mp4


## 8. Summary report

In [18]:
# Print summary table for multi-session QC (sorted chronologically)
from qc_utils import _sort_qc_results_chronologically

outlier_threshold = 50

sorted_results = _sort_qc_results_chronologically(qc_results)

print(f"{'Session':<20} {'Mean dist':>10} {'Median dist':>12} {'Time diff (ms)':>15} {'Dropouts':>10} {'Outlier':>8}")
print('-' * 80)

for r in sorted_results:
    if 'error' in r:
        print(f"{r['session']:<20} ERROR: {r['error']}")
        continue
    s = r['summary']
    t = r['temporal']
    high_dist_frames = np.sum(np.nanmean(r['distances'], axis=1) > 50)
    is_outlier = '*' if s['overall_mean'] > outlier_threshold else ''
    print(f"{r['session']:<20} {s['overall_mean']:>10.1f} {s['overall_median']:>12.1f} "
          f"{t['mean_time_diff_ms']:>15.1f} {high_dist_frames:>10} {is_outlier:>8}")

outlier_sessions = [r['session'] for r in sorted_results
                    if 'error' not in r and r['summary']['overall_mean'] > outlier_threshold]
if outlier_sessions:
    print(f"\n* Outlier sessions (mean dist > {outlier_threshold} units): {', '.join(outlier_sessions)}")

Session               Mean dist  Median dist  Time diff (ms)   Dropouts  Outlier
--------------------------------------------------------------------------------
2025_09_29_2              192.8        128.8             5.0      34723        *
2025_09_30_1              258.1        197.5             5.0      35967        *
2025_10_01_2               36.1         19.3             5.0       5516         
2025_10_01_1              152.2         90.4             5.0      32073        *
2025_10_02_2               27.4         20.2             5.0       1916         
2025_10_02_1              270.4        251.1             5.0      35995        *
2025_10_27_1               14.8         11.6             5.0          8         
2025_10_27_2               14.8         11.9             5.0          0         
2025_10_28_2               11.5          9.0             5.0          0         
2025_10_28_1               11.6          9.8             5.0          1         
2025_10_29_2               1

## 9. Calibration epoch statistical comparison

Tests whether sessions under different calibrations show significantly different SLEAP-DANNCE distances (Kruskal-Wallis + pairwise Mann-Whitney U), and whether distance is correlated with days elapsed since the most recent calibration (Spearman ρ).

In [56]:
# Spearman correlation: distance vs. days since last calibration
days_list, overall_means, labels = compute_days_since_calibration(
    qc_results, calibration_dates
)

fig, ax = plt.subplots(figsize=(8, 5))
_, corr_stats = plot_days_since_calibration(
    days_list, overall_means, labels=labels,
    outlier_threshold=outlier_threshold, rat=rat, ax=ax
)
plt.tight_layout()
plt.show()

print(f"Spearman ρ = {corr_stats['spearman_rho']:.3f}, "
      f"p = {corr_stats['spearman_pvalue']:.4f}, "
      f"n = {corr_stats['n']} sessions")

Spearman ρ = -0.261, p = 0.0279, n = 71 sessions


In [58]:
# Kruskal-Wallis test: does distance differ by calibration epoch?
outlier_threshold = 50

epoch_result = test_calibration_epoch_effect(
    qc_results, calibration_dates, outlier_threshold=outlier_threshold
)

print(f"Kruskal-Wallis H = {epoch_result['kruskal_stat']:.3f}, "
      f"p = {epoch_result['kruskal_pvalue']:.4f}")
print(f"\nSessions per epoch:")
for ep in epoch_result['epoch_order']:
    vals = epoch_result['epoch_data'][ep]
    print(f"  {ep}: n={len(vals)}, mean={np.mean(vals):.2f}, median={np.median(vals):.2f}")

print(f"\nPairwise Mann-Whitney U (post-hoc):")
for (ea, eb), (u, p) in epoch_result['posthoc'].items():
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {ea} vs {eb}: U={u:.0f}, p={p:.4f} {sig}")

fig, ax = plt.subplots(figsize=(10, 5))
plot_calibration_epoch_effect(epoch_result, rat=rat, ax=ax)
plt.tight_layout()
plt.show()

Kruskal-Wallis H = 19.565, p = 0.0006

Sessions per epoch:
  2025_07_27: n=16, mean=16.13, median=12.95
  2025_11_06: n=12, mean=13.20, median=13.17
  2025_11_13: n=27, mean=14.23, median=13.68
  2025_12_08: n=8, mean=15.08, median=14.84
  2026_02_06: n=8, mean=18.51, median=18.36

Pairwise Mann-Whitney U (post-hoc):
  2025_07_27 vs 2025_11_06: U=84, p=0.5934 ns
  2025_07_27 vs 2025_11_13: U=187, p=0.4739 ns
  2025_07_27 vs 2025_12_08: U=20, p=0.0057 **
  2025_07_27 vs 2026_02_06: U=16, p=0.0022 **
  2025_11_06 vs 2025_11_13: U=133, p=0.3858 ns
  2025_11_06 vs 2025_12_08: U=9, p=0.0015 **
  2025_11_06 vs 2026_02_06: U=1, p=0.0000 ***
  2025_11_13 vs 2025_12_08: U=78, p=0.2520 ns
  2025_11_13 vs 2026_02_06: U=24, p=0.0004 ***
  2025_12_08 vs 2026_02_06: U=5, p=0.0030 **


## 10. Snout error spatial density

**Hypothesis**: Snout tracking error is elevated when the animal is at the water port because the nose is occluded from one or more cameras at that location.

**Method**: For each calibration epoch, plot two 2D XY density maps using the DANNCE Snout keypoint position:
- Top row: density of all frames (where the animal spends time)
- Bottom row: enrichment of high-Snout-error frames relative to the all-frames baseline

A concentrated hot spot in the enrichment map that is not present in the all-frames map would support the occlusion hypothesis, especially if it is consistent across calibration epochs (i.e. tied to a fixed location like the water port rather than to any particular calibration).

In [59]:
# Snout XY spatial density: mean error per bin, per calibration epoch
# NOTE: this cell loads raw DANNCE data for each session — may take several minutes

snout_idx = NODES.index('Snout')  # 0

outlier_threshold = 30

fig = plot_snout_error_spatial_density(
    qc_results,
    calibration_dates=calibration_dates,
    snout_idx=snout_idx,
    n_bins=20,
    outlier_threshold=outlier_threshold,
    min_frames_per_bin=5,
)
plt.show()
